In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('../../outputs/df_engineered.csv', 
                  index_col='datetime', 
                  parse_dates=True)

In [4]:
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
print(f'Missing values: {df.isnull().sum().sum()}')

Shape: (8736, 48)
Date range: 2022-01-02 00:30:00+01:00 to 2022-12-31 23:30:00+01:00
Missing values: 0


In [5]:
import statsmodels
print(statsmodels.__version__)

0.14.5


In [6]:
# Use only daytime hours for modelling (nighttime GHI is always 0, adds noise)
ghi = df['GHI']

# Train/test split — 80% train, 20% test (chronological)
split = int(len(ghi) * 0.8)
train = ghi.iloc[:split]
test  = ghi.iloc[split:]

print(f'Train: {len(train)} rows  ({train.index.min().date()} to {train.index.max().date()})')
print(f'Test:  {len(test)} rows   ({test.index.min().date()} to {test.index.max().date()})')

Train: 6988 rows  (2022-01-02 to 2022-10-20)
Test:  1748 rows   (2022-10-20 to 2022-12-31)


In [7]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(train)
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value:       {result[1]:.4f}')
print(f'Stationary:    {result[1] < 0.05}')

ADF Statistic: -6.0573
p-value:       0.0000
Stationary:    True


In [8]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Fit ARIMA(2,0,2) on train
model_arima = ARIMA(train, order=(2, 0, 2))
result_arima = model_arima.fit()

print(result_arima.summary())

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


                               SARIMAX Results                                
Dep. Variable:                    GHI   No. Observations:                 6988
Model:                 ARIMA(2, 0, 2)   Log Likelihood              -41676.512
Date:                Mon, 16 Mar 2026   AIC                          83365.024
Time:                        20:18:07   BIC                          83406.135
Sample:                    01-02-2022   HQIC                         83379.191
                         - 10-20-2022                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        213.9820      6.927     30.893      0.000     200.406     227.558
ar.L1          1.8084      0.011    164.377      0.000       1.787       1.830
ar.L2         -0.8846      0.010    -90.834      0.0

In [9]:
# Forecast on test set
forecast_arima = result_arima.forecast(steps=len(test))

# Clip negative values (GHI cannot be negative)
forecast_arima = forecast_arima.clip(lower=0)

# Evaluate
mae  = mean_absolute_error(test, forecast_arima)
rmse = np.sqrt(mean_squared_error(test, forecast_arima))

print(f'Baseline ARIMA(2,0,2)')
print(f'MAE:  {mae:.2f} W/m²')
print(f'RMSE: {rmse:.2f} W/m²')

Baseline ARIMA(2,0,2)
MAE:  185.62 W/m²
RMSE: 194.99 W/m²
